# GROUP BY, CTEs, and Subqueries

This notebook teaches three powerful SQL concepts through hands-on practice:

1. **GROUP BY** - Aggregate data into summary statistics
2. **CTEs (Common Table Expressions)** - Write readable, modular queries with `WITH` clauses
3. **Subqueries** - Nest queries inside queries for complex logic

We will build from simple aggregations up to a realistic multi-CTE business report.

---

## Setup: Helper Function

We use a `run_query` helper that connects to our PostgreSQL database, executes a query, and prints the results as a formatted table.

In [12]:
import psycopg2
from psycopg2.extras import RealDictCursor
from tabulate import tabulate

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "week2_db",
    "user": "student",
    "password": "student123"
}

def run_query(query, params=None):
    """Execute a SQL query and print results as a formatted table."""
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query, params)
            if cur.description:  # SELECT query
                rows = cur.fetchall()
                if rows:
                    headers = rows[0].keys()
                    print(tabulate([list(r.values()) for r in rows], headers=headers, tablefmt="grid"))
                    print(f"\n({len(rows)} row(s))")
                else:
                    print("No results.")
            else:
                conn.commit()
                print(f"Query executed. Rows affected: {cur.rowcount}")
    finally:
        conn.close()

### Quick Schema Reminder

Let's remind ourselves of the tables we are working with:

In [13]:
run_query("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'company'
ORDER BY table_name;
""")

+--------------+
| table_name   |
+==============+
| departments  |
+--------------+
| employees    |
+--------------+
| products     |
+--------------+
| sales        |
+--------------+

(4 row(s))


In [14]:
run_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'company' AND table_name = 'employees'
ORDER BY ordinal_position;
""")

+---------------+-------------------+
| column_name   | data_type         |
+===============+===================+
| emp_id        | integer           |
+---------------+-------------------+
| first_name    | character varying |
+---------------+-------------------+
| last_name     | character varying |
+---------------+-------------------+
| email         | character varying |
+---------------+-------------------+
| department_id | integer           |
+---------------+-------------------+
| salary        | numeric           |
+---------------+-------------------+
| hire_date     | date              |
+---------------+-------------------+
| manager_id    | integer           |
+---------------+-------------------+
| is_active     | boolean           |
+---------------+-------------------+

(9 row(s))


In [15]:
run_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'company' AND table_name = 'departments'
ORDER BY ordinal_position;
""")

+---------------+-------------------+
| column_name   | data_type         |
+===============+===================+
| dept_id       | integer           |
+---------------+-------------------+
| dept_name     | character varying |
+---------------+-------------------+
| location      | character varying |
+---------------+-------------------+
| budget        | numeric           |
+---------------+-------------------+

(4 row(s))


In [16]:
run_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'company' AND table_name = 'sales'
ORDER BY ordinal_position;
""")

+---------------+-------------------+
| column_name   | data_type         |
+===============+===================+
| sale_id       | integer           |
+---------------+-------------------+
| employee_id   | integer           |
+---------------+-------------------+
| product_id    | integer           |
+---------------+-------------------+
| quantity      | integer           |
+---------------+-------------------+
| sale_date     | date              |
+---------------+-------------------+
| region        | character varying |
+---------------+-------------------+

(6 row(s))


---

## 1. Basic GROUP BY

`GROUP BY` collapses multiple rows into summary rows. Every column in the `SELECT` that is **not** inside an aggregate function (`AVG`, `SUM`, `COUNT`, `MAX`, `MIN`) must appear in the `GROUP BY` clause.

### Example 1a: Average salary per department

In [17]:
run_query("""
SELECT
    department_id,
    ROUND(AVG(salary), 2) AS avg_salary
FROM company.employees
GROUP BY department_id
ORDER BY department_id;
""")

+-----------------+--------------+
|   department_id |   avg_salary |
+=================+==============+
|               1 |      94500   |
+-----------------+--------------+
|               2 |      79800   |
+-----------------+--------------+
|               3 |      86571.4 |
+-----------------+--------------+
|               4 |      83000   |
+-----------------+--------------+
|               5 |      80333.3 |
+-----------------+--------------+
|               6 |      82166.7 |
+-----------------+--------------+
|                 |      60000   |
+-----------------+--------------+

(7 row(s))


**What happened:**
- PostgreSQL grouped all employees by their `department_id`.
- For each group it computed `AVG(salary)`.
- We used `ROUND(..., 2)` to keep two decimal places.

### Example 1b: Count of employees per department

In [18]:
run_query("""
SELECT
    department_id,
    COUNT(*) AS employee_count
FROM company.employees
GROUP BY department_id
ORDER BY department_id;
""")

+-----------------+------------------+
|   department_id |   employee_count |
+=================+==================+
|               1 |                8 |
+-----------------+------------------+
|               2 |                5 |
+-----------------+------------------+
|               3 |                7 |
+-----------------+------------------+
|               4 |                6 |
+-----------------+------------------+
|               5 |                9 |
+-----------------+------------------+
|               6 |                6 |
+-----------------+------------------+
|                 |                1 |
+-----------------+------------------+

(7 row(s))


**Key point:** `COUNT(*)` counts every row in the group, including rows where columns might be NULL.

---

## 2. GROUP BY with JOIN

The `department_id` numbers are not very informative. Let us join with the `departments` table to show the actual department **name**.

In [19]:
run_query("""
SELECT
    d.dept_name,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    COUNT(e.emp_id) AS employee_count
FROM company.employees e
JOIN company.departments d ON e.department_id = d.dept_id
GROUP BY d.dept_name
ORDER BY avg_salary DESC;
""")

+-----------------+--------------+------------------+
| dept_name       |   avg_salary |   employee_count |
+=================+==============+==================+
| Engineering     |      94500   |                8 |
+-----------------+--------------+------------------+
| Finance         |      86571.4 |                7 |
+-----------------+--------------+------------------+
| Marketing       |      83000   |                6 |
+-----------------+--------------+------------------+
| Operations      |      82166.7 |                6 |
+-----------------+--------------+------------------+
| Sales           |      80333.3 |                9 |
+-----------------+--------------+------------------+
| Human Resources |      79800   |                5 |
+-----------------+--------------+------------------+

(6 row(s))


**Notice:**
- We `GROUP BY d.dept_name` because that is the non-aggregated column in our SELECT.
- The join key is `e.department_id = d.dept_id` — the two tables spell the department key differently.
- We use `COUNT(e.emp_id)` instead of `COUNT(*)`. When there are no NULL `emp_id`s the result is the same, but `COUNT(column)` ignores NULLs while `COUNT(*)` does not (we will explore this difference in Section 5).

---

## 3. Multi-column GROUP BY

You can group by **multiple columns** to get finer-grained summaries. Let us compute total sales per region per month.

In [20]:
run_query("""
SELECT
    s.region,
    DATE_TRUNC('month', s.sale_date) AS sale_month,
    ROUND(SUM(s.quantity * p.unit_price), 2) AS total_sales,
    COUNT(*) AS num_transactions
FROM company.sales s
JOIN company.products p ON s.product_id = p.product_id
WHERE s.region IS NOT NULL
GROUP BY s.region, DATE_TRUNC('month', s.sale_date)
ORDER BY s.region, sale_month;
""")

+----------+---------------------------+---------------+--------------------+
| region   | sale_month                |   total_sales |   num_transactions |
+==========+===========================+===============+====================+
| East     | 2025-01-01 00:00:00+00:00 |       5899.98 |                  2 |
+----------+---------------------------+---------------+--------------------+
| East     | 2025-02-01 00:00:00+00:00 |       7000    |                  1 |
+----------+---------------------------+---------------+--------------------+
| East     | 2025-03-01 00:00:00+00:00 |      11400    |                  3 |
+----------+---------------------------+---------------+--------------------+
| East     | 2025-04-01 00:00:00+00:00 |       6799.94 |                  3 |
+----------+---------------------------+---------------+--------------------+
| East     | 2025-05-01 00:00:00+00:00 |      24000    |                  3 |
+----------+---------------------------+---------------+--------

**Key points:**
- The `sales` table stores `quantity` and a `product_id`, **not** a revenue amount. Revenue is computed by joining `products` and multiplying: `s.quantity * p.unit_price`.
- `DATE_TRUNC('month', s.sale_date)` converts any date to the first day of that month, effectively grouping all sales in the same calendar month together.
- We group by **both** `region` and the truncated month, producing one row per unique (region, month) pair.
- The `WHERE s.region IS NOT NULL` filter excludes NULL regions so we get clean output. We will revisit NULL handling in Section 5.

---

## 4. HAVING - Filtering Aggregated Results

`WHERE` filters rows **before** aggregation. `HAVING` filters groups **after** aggregation.

### Find departments with average salary above 82,000

In [21]:
run_query("""
SELECT
    d.dept_name,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    COUNT(e.emp_id) AS employee_count
FROM company.employees e
JOIN company.departments d ON e.department_id = d.dept_id
GROUP BY d.dept_name
HAVING AVG(e.salary) > 82000
ORDER BY avg_salary DESC;
""")

+-------------+--------------+------------------+
| dept_name   |   avg_salary |   employee_count |
+=============+==============+==================+
| Engineering |      94500   |                8 |
+-------------+--------------+------------------+
| Finance     |      86571.4 |                7 |
+-------------+--------------+------------------+
| Marketing   |      83000   |                6 |
+-------------+--------------+------------------+
| Operations  |      82166.7 |                6 |
+-------------+--------------+------------------+

(4 row(s))


**Why HAVING and not WHERE?**

| Clause | When it runs | Works on | Example |
|--------|-------------|----------|--------|
| `WHERE` | Before GROUP BY | Individual rows | `WHERE salary > 50000` |
| `HAVING` | After GROUP BY | Aggregated groups | `HAVING AVG(salary) > 75000` |

You **cannot** write `WHERE AVG(salary) > 75000` because the average does not exist yet at the point WHERE is evaluated.

---

## 5. Three Types of COUNT - Understanding NULLs

The `sales` table has a `region` column that may contain NULLs. Let us see how the three COUNT variants behave differently.

### `COUNT(*)` vs `COUNT(column)` vs `COUNT(DISTINCT column)`

In [22]:
run_query("""
SELECT
    COUNT(*) AS count_star,
    COUNT(region) AS count_region,
    COUNT(DISTINCT region) AS count_distinct_region
FROM company.sales;
""")

+--------------+----------------+-------------------------+
|   count_star |   count_region |   count_distinct_region |
+==============+================+=========================+
|          128 |            115 |                       4 |
+--------------+----------------+-------------------------+

(1 row(s))


In [27]:
# count rows where count_region is Null
run_query("""
SELECT Count(*) AS count_null_region
FROM company.sales
WHERE region IS NULL;
""")

+---------------------+
|   count_null_region |
+=====================+
|                  13 |
+---------------------+

(1 row(s))


### What do these numbers mean?

| Count | Meaning | Includes NULLs? |
|-------|---------|----------------|
| `COUNT(*)` | Total number of rows in the table | Yes |
| `COUNT(region)` | Number of non-NULL values in region | No |
| `COUNT(DISTINCT region)` | Number of unique non-NULL regions | No |

The difference between `COUNT(*)` and `COUNT(region)` tells us **how many rows have a NULL region**. The difference between `COUNT(region)` and `COUNT(DISTINCT region)` tells us how many **duplicate** region values exist.

Let us verify the NULL count:

In [28]:
run_query("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_regions,
    SUM(CASE WHEN region IS NOT NULL THEN 1 ELSE 0 END) AS non_null_regions
FROM company.sales;
""")

+--------------+----------------+--------------------+
|   total_rows |   null_regions |   non_null_regions |
+==============+================+====================+
|          128 |             13 |                115 |
+--------------+----------------+--------------------+

(1 row(s))


---

## 6. Simple CTE (Common Table Expression)

A CTE lets you define a named temporary result set using `WITH`. It makes queries much more readable than deeply nested subqueries.

### Problem: Find departments where the average salary is above the company-wide average

**Step 1:** Compute the company-wide average salary.

In [ ]:
run_query("""
SELECT ROUND(AVG(salary), 2) AS company_avg_salary
FROM company.employees;
""")

**Step 2:** Now use a CTE to combine both steps into one clean query.

In [29]:
run_query("""
WITH company_avg AS (
    SELECT AVG(salary) AS avg_sal
    FROM company.employees
)
SELECT
    d.dept_name,
    ROUND(AVG(e.salary), 2) AS dept_avg_salary,
    ROUND(ca.avg_sal, 2) AS company_avg_salary
FROM company.employees e
JOIN company.departments d ON e.department_id = d.dept_id
CROSS JOIN company_avg ca
GROUP BY d.dept_name, ca.avg_sal
HAVING AVG(e.salary) > ca.avg_sal
ORDER BY dept_avg_salary DESC;
""")

+-------------+-------------------+----------------------+
| dept_name   |   dept_avg_salary |   company_avg_salary |
+=============+===================+======================+
| Engineering |           94500   |              84166.7 |
+-------------+-------------------+----------------------+
| Finance     |           86571.4 |              84166.7 |
+-------------+-------------------+----------------------+

(2 row(s))


**Why is this better?** Without the CTE you would need a nested subquery inside HAVING:

```sql
HAVING AVG(e.salary) > (SELECT AVG(salary) FROM company.employees)
```

The CTE names the concept ("company average") making the query self-documenting.

---

## 7. Multiple CTEs Chained Together

You can define **multiple CTEs** separated by commas. Each CTE can reference earlier CTEs. This is like building a pipeline.

### Problem: Get employee names from the top 3 departments by average salary

**Pipeline:**
1. CTE `dept_stats`: Compute average salary per department
2. CTE `top_departments`: Filter to the top 3 by average salary
3. Main query: Join back to employees to get names

In [30]:
run_query("""
WITH dept_stats AS (
    -- Step 1: Compute department-level statistics
    SELECT
        d.dept_id,
        d.dept_name,
        ROUND(AVG(e.salary), 2) AS avg_salary,
        COUNT(e.emp_id) AS employee_count
    FROM company.employees e
    JOIN company.departments d ON e.department_id = d.dept_id
    GROUP BY d.dept_id, d.dept_name
),
top_departments AS (
    -- Step 2: Keep only the top 3 departments
    SELECT
        dept_id,
        dept_name,
        avg_salary,
        employee_count
    FROM dept_stats
    ORDER BY avg_salary DESC
    LIMIT 3
)
-- Step 3: Get all employees in those top departments
SELECT
    td.dept_name,
    e.first_name || ' ' || e.last_name AS full_name,
    e.salary,
    td.avg_salary AS dept_avg_salary
FROM top_departments td
JOIN company.employees e ON td.dept_id = e.department_id
ORDER BY td.dept_name, e.salary DESC;
""")

+-------------+---------------+----------+-------------------+
| dept_name   | full_name     |   salary |   dept_avg_salary |
+=============+===============+==========+===================+
| Engineering | Alice Chen    |   145000 |           94500   |
+-------------+---------------+----------+-------------------+
| Engineering | Bob Martinez  |   125000 |           94500   |
+-------------+---------------+----------+-------------------+
| Engineering | Carol Johnson |    95000 |           94500   |
+-------------+---------------+----------+-------------------+
| Engineering | David Kim     |    88000 |           94500   |
+-------------+---------------+----------+-------------------+
| Engineering | Eva Patel     |    88000 |           94500   |
+-------------+---------------+----------+-------------------+
| Engineering | Frank Wilson  |    75000 |           94500   |
+-------------+---------------+----------+-------------------+
| Engineering | Grace Lee     |    72000 |           94

**Key insight:** Each CTE is like a variable in Python. You compute it once, name it, and reuse the name. This is far more readable than nesting subqueries three levels deep.

---

## 8. Correlated Subquery in WHERE

A **correlated subquery** references columns from the outer query. It runs once **per row** of the outer query.

### Problem: Find employees who earn more than the average salary of their own department

In [31]:
run_query("""
SELECT
    e.first_name || ' ' || e.last_name AS full_name,
    e.department_id,
    e.salary
FROM company.employees e
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM company.employees e2
    WHERE e2.department_id = e.department_id
)
ORDER BY e.department_id, e.salary DESC;
""")

+----------------+-----------------+----------+
| full_name      |   department_id |   salary |
+================+=================+==========+
| Alice Chen     |               1 |   145000 |
+----------------+-----------------+----------+
| Bob Martinez   |               1 |   125000 |
+----------------+-----------------+----------+
| Carol Johnson  |               1 |    95000 |
+----------------+-----------------+----------+
| Iris Taylor    |               2 |   115000 |
+----------------+-----------------+----------+
| Jack Anderson  |               2 |    85000 |
+----------------+-----------------+----------+
| Nathan Harris  |               3 |   135000 |
+----------------+-----------------+----------+
| Olivia Clark   |               3 |   105000 |
+----------------+-----------------+----------+
| Uma King       |               4 |   120000 |
+----------------+-----------------+----------+
| Victor Wright  |               4 |    95000 |
+----------------+-----------------+----

In [32]:
# writing without using CTEs

# cte version
run_query("""
WITH dept_avg AS (
    SELECT department_id, AVG(salary) AS avg_salary
    FROM company.employees
    GROUP BY department_id
)
SELECT
    e.first_name || ' ' || e.last_name AS full_name,
    e.department_id,
    e.salary
FROM company.employees e
JOIN dept_avg da ON e.department_id = da.department_id
WHERE e.salary > da.avg_salary
ORDER BY e.department_id, e.salary DESC;
""")

+----------------+-----------------+----------+
| full_name      |   department_id |   salary |
+================+=================+==========+
| Alice Chen     |               1 |   145000 |
+----------------+-----------------+----------+
| Bob Martinez   |               1 |   125000 |
+----------------+-----------------+----------+
| Carol Johnson  |               1 |    95000 |
+----------------+-----------------+----------+
| Iris Taylor    |               2 |   115000 |
+----------------+-----------------+----------+
| Jack Anderson  |               2 |    85000 |
+----------------+-----------------+----------+
| Nathan Harris  |               3 |   135000 |
+----------------+-----------------+----------+
| Olivia Clark   |               3 |   105000 |
+----------------+-----------------+----------+
| Uma King       |               4 |   120000 |
+----------------+-----------------+----------+
| Victor Wright  |               4 |    95000 |
+----------------+-----------------+----

**How it works:**

For each employee row, the subquery computes the average salary **of that employee's department** (`WHERE e2.department_id = e.department_id`). If the employee's own salary exceeds that department average, the row is included.

This is a **correlated** subquery because the inner query references `e.department_id` from the outer query. It cannot run independently.

**Performance note:** Correlated subqueries can be slow on large tables because the inner query runs once per outer row. In some cases, rewriting with a CTE or JOIN is faster.

---

## 9. Subquery in FROM (Derived Table)

You can use a subquery in the `FROM` clause as if it were a table. This is called a **derived table**.

### Problem: For each department, show the department name and how far its average salary deviates from the company average

In [33]:
run_query("""
SELECT
    dept_avgs.dept_name,
    ROUND(dept_avgs.avg_dept_salary, 2) AS avg_dept_salary,
    ROUND(company_avg.avg_salary, 2) AS company_avg,
    ROUND(dept_avgs.avg_dept_salary - company_avg.avg_salary, 2) AS deviation
FROM (
    SELECT
        d.dept_name,
        AVG(e.salary) AS avg_dept_salary
    FROM company.employees e
    JOIN company.departments d ON e.department_id = d.dept_id
    GROUP BY d.dept_name
) AS dept_avgs
CROSS JOIN (
    SELECT AVG(salary) AS avg_salary
    FROM company.employees
) AS company_avg
ORDER BY deviation DESC;
""")

+-----------------+-------------------+---------------+-------------+
| dept_name       |   avg_dept_salary |   company_avg |   deviation |
+=================+===================+===============+=============+
| Engineering     |           94500   |       84166.7 |    10333.3  |
+-----------------+-------------------+---------------+-------------+
| Finance         |           86571.4 |       84166.7 |     2404.76 |
+-----------------+-------------------+---------------+-------------+
| Marketing       |           83000   |       84166.7 |    -1166.67 |
+-----------------+-------------------+---------------+-------------+
| Operations      |           82166.7 |       84166.7 |    -2000    |
+-----------------+-------------------+---------------+-------------+
| Sales           |           80333.3 |       84166.7 |    -3833.33 |
+-----------------+-------------------+---------------+-------------+
| Human Resources |           79800   |       84166.7 |    -4366.67 |
+-----------------+-

**Why use a subquery in FROM?** Sometimes you need to aggregate first, then do further computation on those aggregates. The subquery produces an intermediate result that the outer query then works with.

We could rewrite this with CTEs (see next section) for even better readability.

---

## 10. Realistic Business Question - Built Step by Step

### The Request

Management wants a single report showing, **for each department**:
- Department name
- Number of employees
- Average salary
- Total sales revenue (from employees in that department)
- Name of the highest-paid employee

We will build this with multiple CTEs, then show the equivalent nested subquery approach.

### Approach: Multiple CTEs (Recommended for Readability)

In [34]:
run_query("""
WITH dept_salary AS (
    -- CTE 1: Department-level salary statistics
    SELECT
        d.dept_id,
        d.dept_name,
        COUNT(e.emp_id) AS employee_count,
        ROUND(AVG(e.salary), 2) AS avg_salary,
        ROUND(SUM(e.salary), 2) AS total_salary
    FROM company.departments d
    LEFT JOIN company.employees e ON d.dept_id = e.department_id
    GROUP BY d.dept_id, d.dept_name
),
dept_sales AS (
    -- CTE 2: Total sales revenue per department (quantity x unit price)
    SELECT
        e.department_id,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
    FROM company.sales s
    JOIN company.employees e ON s.employee_id = e.emp_id
    JOIN company.products p ON s.product_id = p.product_id
    GROUP BY e.department_id
),
top_earners AS (
    -- CTE 3: Find the highest-paid employee per department
    SELECT DISTINCT ON (e.department_id)
        e.department_id,
        e.first_name || ' ' || e.last_name AS top_earner_name,
        e.salary AS top_earner_salary
    FROM company.employees e
    ORDER BY e.department_id, e.salary DESC, e.emp_id
)
-- Final query: Combine all CTEs
SELECT
    ds.dept_name,
    ds.employee_count,
    ds.avg_salary,
    COALESCE(dv.total_revenue, 0) AS total_revenue,
    te.top_earner_name,
    te.top_earner_salary
FROM dept_salary ds
LEFT JOIN dept_sales dv ON ds.dept_id = dv.department_id
LEFT JOIN top_earners te ON ds.dept_id = te.department_id
ORDER BY ds.avg_salary DESC;
""")

+-----------------+------------------+--------------+-----------------+-------------------+---------------------+
| dept_name       |   employee_count |   avg_salary |   total_revenue | top_earner_name   |   top_earner_salary |
+=================+==================+==============+=================+===================+=====================+
| Engineering     |                8 |      94500   |         3599.94 | Alice Chen        |              145000 |
+-----------------+------------------+--------------+-----------------+-------------------+---------------------+
| Finance         |                7 |      86571.4 |         2299.97 | Nathan Harris     |              135000 |
+-----------------+------------------+--------------+-----------------+-------------------+---------------------+
| Marketing       |                6 |      83000   |        45900    | Uma King          |              120000 |
+-----------------+------------------+--------------+-----------------+-----------------

**Why CTEs shine here:**

Each CTE solves one sub-problem:
- `dept_salary` handles employee counts and averages
- `dept_sales` handles revenue aggregation
- `top_earners` finds the highest-paid person using `DISTINCT ON`
- The final query simply joins the three results

Now let us see what the same query looks like with nested subqueries...

### Alternative: Nested Subquery Approach (Harder to Read)

In [35]:
run_query("""
SELECT
    d.dept_name,
    COALESCE(emp_stats.employee_count, 0) AS employee_count,
    emp_stats.avg_salary,
    COALESCE(sales_rev.total_revenue, 0) AS total_revenue,
    top_emp.top_earner_name,
    top_emp.top_earner_salary
FROM company.departments d
LEFT JOIN (
    SELECT
        department_id,
        COUNT(*) AS employee_count,
        ROUND(AVG(salary), 2) AS avg_salary
    FROM company.employees
    GROUP BY department_id
) emp_stats ON d.dept_id = emp_stats.department_id
LEFT JOIN (
    SELECT
        e.department_id,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
    FROM company.sales s
    JOIN company.employees e ON s.employee_id = e.emp_id
    JOIN company.products p ON s.product_id = p.product_id
    GROUP BY e.department_id
) sales_rev ON d.dept_id = sales_rev.department_id
LEFT JOIN (
    SELECT DISTINCT ON (department_id)
        department_id,
        first_name || ' ' || last_name AS top_earner_name,
        salary AS top_earner_salary
    FROM company.employees
    ORDER BY department_id, salary DESC, emp_id
) top_emp ON d.dept_id = top_emp.department_id
ORDER BY emp_stats.avg_salary DESC NULLS LAST;
""")

+-----------------+------------------+--------------+-----------------+-------------------+---------------------+
| dept_name       |   employee_count |   avg_salary |   total_revenue | top_earner_name   |   top_earner_salary |
+=================+==================+==============+=================+===================+=====================+
| Engineering     |                8 |      94500   |         3599.94 | Alice Chen        |              145000 |
+-----------------+------------------+--------------+-----------------+-------------------+---------------------+
| Finance         |                7 |      86571.4 |         2299.97 | Nathan Harris     |              135000 |
+-----------------+------------------+--------------+-----------------+-------------------+---------------------+
| Marketing       |                6 |      83000   |        45900    | Uma King          |              120000 |
+-----------------+------------------+--------------+-----------------+-----------------

### CTE vs Nested Subqueries: Comparison

| Aspect | CTE Approach | Nested Subquery Approach |
|--------|-------------|------------------------|
| **Readability** | High - each piece is named | Low - logic is buried in indentation |
| **Debugging** | Easy - test each CTE separately | Hard - must extract subqueries manually |
| **Reusability** | A CTE can be referenced multiple times | Each subquery is inlined |
| **Performance** | Similar (PostgreSQL often inlines CTEs) | Similar |
| **Preference** | Recommended for complex queries | Fine for simple, short subqueries |

**Rule of thumb:** If your subquery is more than 3-4 lines, consider a CTE.

---

## 11. Try It Yourself

Now it is your turn. Attempt each exercise before revealing the solution.

---

### Exercise 1: Monthly Revenue by Region with HAVING

Write a query that shows total sales revenue per region per month, but **only** include region-month combinations where total revenue exceeds 20,000. Order by total revenue descending.

*Hint: revenue is not a column — compute it as `s.quantity * p.unit_price` by joining `company.products`.*

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
SELECT
    s.region,
    DATE_TRUNC('month', s.sale_date) AS sale_month,
    ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
FROM company.sales s
JOIN company.products p ON s.product_id = p.product_id
WHERE s.region IS NOT NULL
GROUP BY s.region, DATE_TRUNC('month', s.sale_date)
HAVING SUM(s.quantity * p.unit_price) > 20000
ORDER BY total_revenue DESC;
```

</details>

In [36]:
# Write your solution for Exercise 1 here
run_query("""
-- Your query here
SELECT
    s.region,
    DATE_TRUNC('month', s.sale_date) AS sale_month,
    ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
FROM company.sales s
JOIN company.products p ON s.product_id = p.product_id
WHERE s.region IS NOT NULL
GROUP BY s.region, DATE_TRUNC('month', s.sale_date)
HAVING SUM(s.quantity * p.unit_price) > 20000
ORDER BY total_revenue DESC;
""")

+----------+---------------------------+-----------------+
| region   | sale_month                |   total_revenue |
+==========+===========================+=================+
| West     | 2025-07-01 00:00:00+00:00 |           44600 |
+----------+---------------------------+-----------------+
| South    | 2025-11-01 00:00:00+00:00 |           42000 |
+----------+---------------------------+-----------------+
| North    | 2025-09-01 00:00:00+00:00 |           36500 |
+----------+---------------------------+-----------------+
| North    | 2025-05-01 00:00:00+00:00 |           32600 |
+----------+---------------------------+-----------------+
| West     | 2025-12-01 00:00:00+00:00 |           32100 |
+----------+---------------------------+-----------------+
| North    | 2025-12-01 00:00:00+00:00 |           28500 |
+----------+---------------------------+-----------------+
| North    | 2025-08-01 00:00:00+00:00 |           25500 |
+----------+---------------------------+----------------

---

### Exercise 2: Employees Above Company Average Using CTE

Write a CTE-based query that lists all employees whose salary is above the company-wide average. Show their name, department name, salary, and how much above the company average they earn.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
WITH company_avg AS (
    SELECT AVG(salary) AS avg_sal
    FROM company.employees
)
SELECT
    e.first_name || ' ' || e.last_name AS full_name,
    d.dept_name,
    e.salary,
    ROUND(e.salary - ca.avg_sal, 2) AS above_average_by
FROM company.employees e
JOIN company.departments d ON e.department_id = d.dept_id
CROSS JOIN company_avg ca
WHERE e.salary > ca.avg_sal
ORDER BY e.salary DESC;
```

</details>

In [37]:
# Write your solution for Exercise 2 here
run_query("""
-- Your query here
WITH company_avg AS (
    SELECT AVG(salary) AS avg_sal
    FROM company.employees
)
SELECT
    e.first_name || ' ' || e.last_name AS full_name,
    d.dept_name,
    e.salary,
    ROUND(e.salary - ca.avg_sal, 2) AS above_average_by
FROM company.employees e
JOIN company.departments d ON e.department_id = d.dept_id
CROSS JOIN company_avg ca
WHERE e.salary > ca.avg_sal
ORDER BY e.salary DESC;
""")

+----------------+-----------------+----------+--------------------+
| full_name      | dept_name       |   salary |   above_average_by |
+================+=================+==========+====================+
| Alice Chen     | Engineering     |   145000 |           60833.3  |
+----------------+-----------------+----------+--------------------+
| Nathan Harris  | Finance         |   135000 |           50833.3  |
+----------------+-----------------+----------+--------------------+
| Amy Adams      | Sales           |   130000 |           45833.3  |
+----------------+-----------------+----------+--------------------+
| James Campbell | Operations      |   125000 |           40833.3  |
+----------------+-----------------+----------+--------------------+
| Bob Martinez   | Engineering     |   125000 |           40833.3  |
+----------------+-----------------+----------+--------------------+
| Uma King       | Marketing       |   120000 |           35833.3  |
+----------------+----------------

In [53]:
# Write your solution for Exercise 2 here
run_query("""
-- Your query here
SELECT
    e.first_name || ' ' || e.last_name AS full_name,
    d.dept_name,
    e.salary,
    ROUND(e.salary - (SELECT AVG(salary) FROM company.employees), 2) AS above_average_by
FROM company.employees e
JOIN company.departments d ON e.department_id = d.dept_id
WHERE e.salary > (SELECT AVG(salary) FROM company.employees)
ORDER BY e.salary DESC;
""")

+----------------+-----------------+----------+--------------------+
| full_name      | dept_name       |   salary |   above_average_by |
+================+=================+==========+====================+
| Alice Chen     | Engineering     |   145000 |           60833.3  |
+----------------+-----------------+----------+--------------------+
| Nathan Harris  | Finance         |   135000 |           50833.3  |
+----------------+-----------------+----------+--------------------+
| Amy Adams      | Sales           |   130000 |           45833.3  |
+----------------+-----------------+----------+--------------------+
| James Campbell | Operations      |   125000 |           40833.3  |
+----------------+-----------------+----------+--------------------+
| Bob Martinez   | Engineering     |   125000 |           40833.3  |
+----------------+-----------------+----------+--------------------+
| Uma King       | Marketing       |   120000 |           35833.3  |
+----------------+----------------

---

### Exercise 3: Department with Most Revenue

Write a query that finds the department with the **highest total sales revenue**. Show the department name, total revenue, and the number of sales transactions. Use CTEs.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
WITH dept_revenue AS (
    SELECT
        d.dept_name,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue,
        COUNT(s.sale_id) AS num_transactions
    FROM company.sales s
    JOIN company.employees e ON s.employee_id = e.emp_id
    JOIN company.products p ON s.product_id = p.product_id
    JOIN company.departments d ON e.department_id = d.dept_id
    GROUP BY d.dept_name
)
SELECT *
FROM dept_revenue
ORDER BY total_revenue DESC
LIMIT 1;
```

</details>

In [38]:
# Write your solution for Exercise 3 here
run_query("""
-- Your query here
WITH dept_revenue AS (
    SELECT
        d.dept_name,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue,
        COUNT(s.sale_id) AS num_transactions
    FROM company.sales s
    JOIN company.employees e ON s.employee_id = e.emp_id
    JOIN company.products p ON s.product_id = p.product_id
    JOIN company.departments d ON e.department_id = d.dept_id
    GROUP BY d.dept_name
)
SELECT *
FROM dept_revenue
ORDER BY total_revenue DESC
LIMIT 1;
""")

+-------------+-----------------+--------------------+
| dept_name   |   total_revenue |   num_transactions |
+=============+=================+====================+
| Sales       |          629749 |                109 |
+-------------+-----------------+--------------------+

(1 row(s))


---

## Summary Cheat Sheet

| Concept | Syntax | Use when |
|---------|--------|----------|
| **GROUP BY** | `SELECT col, AGG(x) FROM t GROUP BY col` | You need summary statistics per category |
| **HAVING** | `GROUP BY ... HAVING AGG(x) > value` | You need to filter after aggregation |
| **COUNT(*)** | `COUNT(*)` | Count all rows including NULLs |
| **COUNT(col)** | `COUNT(column_name)` | Count only non-NULL values |
| **COUNT DISTINCT** | `COUNT(DISTINCT col)` | Count unique non-NULL values |
| **CTE** | `WITH name AS (query) SELECT ... FROM name` | You want readable, modular queries |
| **Correlated subquery** | `WHERE col > (SELECT ... FROM t2 WHERE t2.x = t1.x)` | Per-row comparison with a related subset |
| **Derived table** | `FROM (SELECT ...) AS alias` | You need to aggregate then further process |

**Next steps:** Practice these patterns on your own datasets. The more you write GROUP BY and CTE queries, the more natural they become.